# 03. 심화 — 재시도, 실패 정책, 호출 예산과 평가

그래프 모양만 선언했다고 production-ready가 되는 것은 아닙니다. 이 실습은 실제
ADK RetryConfig를 확인하고, fan-out 결과의 실패 정책·호출 예산·워크플로 선택 기준을
결정론적으로 테스트합니다.

## 학습 목표

1. 일시적 오류를 제한 횟수만 재시도하고 오류 Event를 관찰한다.
2. Join 이후 필수 분기와 선택 분기의 실패를 다르게 처리한다.
3. 모델 호출 예산을 실행 전에 차단한다.
4. 정적 그래프, 동적 workflow, 자율 Agent의 경계를 사례로 판정한다.

모든 셀은 API 키 없이 실행됩니다. 운영 적용 전에는 공식 문서와 실제 배포 버전을
다시 확인하세요.


In [ ]:
from importlib.metadata import version

assert version("google-adk") == "2.8.0", (
    "이 노트북은 google-adk==2.8.0으로 검증했습니다."
)


## 1. ADK 노드 재시도와 timeout

실패를 무한 반복하지 않도록 max_attempts를 고정하고, 지연 상한과 jitter 정책을
명시합니다. 아래 노드는 첫 호출에 ConnectionError를 내고 두 번째 호출에
성공하도록 만든 통제된 실험입니다. 재시도 중 발생한 오류도 Event로 남는다는 점을
확인합니다.


In [ ]:
import logging

from google.adk import Event, Runner, Workflow
from google.adk.sessions import InMemorySessionService
from google.adk.workflow import RetryConfig, node
from google.genai import types


attempt_counter = {"count": 0}


@node(
    retry_config=RetryConfig(
        max_attempts=3,
        initial_delay=0.01,
        max_delay=0.01,
        backoff_factor=1.0,
        jitter=0.0,
        exceptions=[ConnectionError],
    ),
    timeout=0.2,
)
async def flaky_weather_source(node_input):
    attempt_counter["count"] += 1
    if attempt_counter["count"] == 1:
        raise ConnectionError("통제된 일시 오류")
    return Event(
        output={
            "status": "ok",
            "attempts": attempt_counter["count"],
            "temp_f": 72,
        }
    )


retry_workflow = Workflow(
    name="retry_workflow",
    edges=[("START", flaky_weather_source)],
)


In [ ]:
async def run_retry_demo():
    service = InMemorySessionService()
    session = await service.create_session(
        app_name="graph_learning_lab",
        user_id="operator",
        session_id="advanced-retry",
    )
    runner = Runner(
        node=retry_workflow,
        app_name="graph_learning_lab",
        session_service=service,
    )
    message = types.Content(
        role="user",
        parts=[types.Part(text="재시도 실험")],
    )

    observed = []
    # ADK가 예상된 첫 실패의 traceback을 기록하므로 실습 출력에서는 로그만 억제합니다.
    # Event 자체는 억제하지 않아 아래에서 실패와 회복을 모두 검사합니다.
    logging.disable(logging.CRITICAL)
    try:
        async for event in runner.run_async(
            user_id="operator",
            session_id=session.id,
            new_message=message,
        ):
            observed.append(
                {
                    "output": event.output,
                    "error_code": event.error_code,
                    "error_message": event.error_message,
                }
            )
    finally:
        logging.disable(logging.NOTSET)
    return observed


retry_events = await run_retry_demo()
assert attempt_counter["count"] == 2
assert retry_events[0]["error_code"] == "ConnectionError"
assert retry_events[-1]["output"]["status"] == "ok"
retry_events


재시도가 성공해도 앞선 오류 Event는 관측 기록에 남습니다. 따라서 모니터링은
개별 오류 Event와 invocation의 최종 성공/실패를 구분해야 합니다. 재시도 대상
예외를 좁게 지정하고, 인증 오류나 잘못된 schema처럼 반복해도 낫지 않는 오류는
즉시 실패시키는 편이 안전합니다.


## 2. Join 이후 실패 정책

날씨와 체력은 안전한 전략에 필수, 코스 부가정보는 선택이라고 가정합니다.
필수 분기가 실패하면 BLOCKED, 선택 분기만 실패하면 DEGRADED로 진행합니다.
이 정책은 업무 요구사항이며 JoinNode가 자동으로 결정해 주지 않습니다.


In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class BranchOutcome:
    name: str
    ok: bool
    data: dict[str, Any] | None = None
    error: str | None = None


REQUIRED_BRANCHES = {"weather", "fitness"}


def apply_join_policy(outcomes):
    by_name = {outcome.name: outcome for outcome in outcomes}
    missing_required = sorted(
        name
        for name in REQUIRED_BRANCHES
        if name not in by_name or not by_name[name].ok
    )
    errors = {
        name: outcome.error
        for name, outcome in by_name.items()
        if not outcome.ok
    }
    payload = {
        name: outcome.data
        for name, outcome in by_name.items()
        if outcome.ok
    }

    if missing_required:
        status = "BLOCKED"
    elif errors:
        status = "DEGRADED"
    else:
        status = "READY"

    return {
        "status": status,
        "payload": payload,
        "errors": errors,
        "missing_required": missing_required,
    }


all_ok = [
    BranchOutcome("weather", True, {"temp_f": 86}),
    BranchOutcome("course", True, {"elevation_m": 220}),
    BranchOutcome("fitness", True, {"weekly_km": 45}),
]
optional_failure = [
    BranchOutcome("weather", True, {"temp_f": 86}),
    BranchOutcome("course", False, error="course API timeout"),
    BranchOutcome("fitness", True, {"weekly_km": 45}),
]
required_failure = [
    BranchOutcome("weather", False, error="weather API unavailable"),
    BranchOutcome("course", True, {"elevation_m": 220}),
    BranchOutcome("fitness", True, {"weekly_km": 45}),
]

policy_results = {
    "all_ok": apply_join_policy(all_ok),
    "optional_failure": apply_join_policy(optional_failure),
    "required_failure": apply_join_policy(required_failure),
}
assert policy_results["all_ok"]["status"] == "READY"
assert policy_results["optional_failure"]["status"] == "DEGRADED"
assert policy_results["required_failure"]["status"] == "BLOCKED"
policy_results


## 3. route와 모델 호출 예산을 함께 강제하기

BLOCKED인 입력에는 모델을 호출해 그럴듯한 답을 만들지 않고 수동 검토로 보냅니다.
READY 또는 DEGRADED인 입력만 온도 router를 통과하며 호출 예산 1회를 예약합니다.


In [ ]:
@dataclass
class CallBudget:
    maximum: int
    used: int = 0

    def reserve(self, amount=1):
        if self.used + amount > self.maximum:
            raise RuntimeError("LLM 호출 예산 초과")
        self.used += amount


def choose_route(policy_result):
    if policy_result["status"] == "BLOCKED":
        return "MANUAL_REVIEW"

    temperature = policy_result["payload"]["weather"]["temp_f"]
    if temperature >= 80:
        return "HOT"
    if temperature <= 40:
        return "COLD"
    return "NORMAL"


def plan_execution(outcomes, maximum_calls=1):
    policy = apply_join_policy(outcomes)
    route = choose_route(policy)
    budget = CallBudget(maximum=maximum_calls)
    if route != "MANUAL_REVIEW":
        budget.reserve(1)
    return {
        "status": policy["status"],
        "route": route,
        "model_calls": budget.used,
    }


assert plan_execution(all_ok) == {
    "status": "READY",
    "route": "HOT",
    "model_calls": 1,
}
assert plan_execution(optional_failure) == {
    "status": "DEGRADED",
    "route": "HOT",
    "model_calls": 1,
}
assert plan_execution(required_failure) == {
    "status": "BLOCKED",
    "route": "MANUAL_REVIEW",
    "model_calls": 0,
}

try:
    plan_execution(all_ok, maximum_calls=0)
except RuntimeError as error:
    budget_guard = str(error)
else:
    raise AssertionError("호출 예산 0에서 실행이 차단되어야 합니다.")

budget_guard


## 4. 정적 그래프, 동적 workflow, 자율 Agent 고르기

영상의 마지막 판단 기준을 운영 관점으로 조금 더 엄격하게 표현합니다. 입력 전에
노드와 엣지를 그릴 수 있으면 정적 그래프, 런타임 데이터가 작업 폭이나 반복 횟수를
정하면 동적 workflow, 목표 달성 절차 자체를 모델이 탐색해야 하면 자율 Agent가
후보입니다. 실제 시스템은 이 셋을 중첩할 수 있습니다.


In [ ]:
def recommend_control_flow(*, shape_changes_at_runtime, open_ended_planning):
    if open_ended_planning:
        return "autonomous_agent"
    if shape_changes_at_runtime:
        return "dynamic_workflow"
    return "static_graph"


decision_cases = [
    {
        "case": "날씨·코스·체력 조회 후 고정 전략 선택",
        "shape_changes_at_runtime": False,
        "open_ended_planning": False,
        "expected": "static_graph",
    },
    {
        "case": "입력 파일 수만큼 검증 노드를 런타임 생성",
        "shape_changes_at_runtime": True,
        "open_ended_planning": False,
        "expected": "dynamic_workflow",
    },
    {
        "case": "미지의 사이트에서 조사 절차와 도구를 스스로 선택",
        "shape_changes_at_runtime": True,
        "open_ended_planning": True,
        "expected": "autonomous_agent",
    },
]

for case in decision_cases:
    actual = recommend_control_flow(
        shape_changes_at_runtime=case["shape_changes_at_runtime"],
        open_ended_planning=case["open_ended_planning"],
    )
    assert actual == case["expected"]
    case["actual"] = actual

decision_cases


## 5. 배포 전 체크리스트

- **계약**: 각 노드의 입력·출력 schema, 단위, nullable 필드와 버전을 고정한다.
- **실패**: timeout, 좁은 retry 조건, 최대 시도, backoff, fallback과 중단 정책을 둔다.
- **Join**: 필수/선택 분기, 부분 성공, 늦은 응답과 중복 결과 처리 방식을 정한다.
- **상태**: 병렬 분기는 같은 state key를 쓰지 않고, 대용량 자료는 artifact에 둔다.
- **내구성**: 메모리 session 대신 운영 저장소와 checkpoint·idempotency를 설계한다.
- **관측성**: invocation/node trace, 지연, retry, route, token과 비용을 함께 기록한다.
- **보안**: 최소 권한, secret manager, 입력 검증, 도구 allowlist와 감사 로그를 둔다.
- **평가**: 최종 문장뿐 아니라 선택 route와 tool trajectory를 회귀 테스트한다.
- **비용**: 호출 수, token 상한, 동시성, 외부 API와 네트워크 비용도 예산화한다.
- **버전**: ADK와 모델 버전을 고정하고 upgrade 시 graph·event schema를 재검증한다.

이제 [README.md](README.md)의 상세 해설과
[translation.ko.md](translation.ko.md)의 영상 챕터별 번역·해설을 함께 읽어
개념, 구현, 운영상의 한계를 연결해 보세요.
